# Track 2 — Intelligent Well Archive

Optional **local inspection and keyword-retrieval baseline**, not a finished question-answering system. PDF parsing, evidence-backed answers and a reproducible solution notebook are participant responsibilities. Use the approved dataset provided by the organisers. The notebook locates it when run from the repository root or this starter-kit directory; otherwise set `DATA_ROOT` to its location.

In [ ]:
from pathlib import Path
import csv
import re
from html import unescape

# Resolve data from either the repository root or the starter-kit directory.
def find_data_root():
    candidates = [
        Path.cwd() / "data" / "track_2",
        Path.cwd() / "../../data/track_2",
        Path.cwd() / "../data/track_2",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate data/track_2. Run from the repository root or the track starter-kit "
        "directory, or set DATA_ROOT to the approved dataset location."
    )

DATA_ROOT = find_data_root()
MANIFEST = DATA_ROOT / "metadata" / "document_manifest.csv"
DOCUMENTS = DATA_ROOT / "documents"
EXPECTED_COLUMNS = ["question_id", "answer", "evidence", "confidence"]
QUESTIONS = DATA_ROOT / "open_test" / "open_test_questions.csv"
with QUESTIONS.open(newline="", encoding="utf-8-sig") as handle:
    questions = list(csv.DictReader(handle))
print("Published questions:", len(questions))
print("Dataset found:", DATA_ROOT.exists())
print("Manifest found:", MANIFEST.exists())
print("Documents folder found:", DOCUMENTS.exists())

In [ ]:
if MANIFEST.is_file():
    with MANIFEST.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        print("Manifest columns:", reader.fieldnames)
        print("Sample manifest rows:", [row for _, row in zip(range(3), reader)])
else:
    print("Manifest unavailable at placeholder path; confirm approved data root with technical support.")

In [ ]:
def read_local_text(path):
    if path.suffix.lower() not in {".txt", ".html", ".htm", ".xml"}:
        raise ValueError("This basic reader supports only text-like files; add a PDF parser separately.")
    raw = path.read_text(encoding="utf-8", errors="replace")
    if path.suffix.lower() in {".html", ".htm", ".xml"}:
        raw = unescape(re.sub(r"<[^>]+>", " ", raw))
    return raw

files = sorted(
    p for p in DOCUMENTS.rglob("*")
    if p.is_file() and p.suffix.lower() in {".txt", ".html", ".htm", ".xml"}
) if DOCUMENTS.is_dir() else []
print("Readable text-like files:", len(files))
if files:
    print("Preview from", files[0].relative_to(DATA_ROOT), ":", read_local_text(files[0])[:350])
print("PDF content is not covered by this baseline; use an approved local PDF parser.")

In [ ]:
def keyword_search(question, paths, top_k=3):
    stopwords = {"the", "a", "an", "in", "on", "of", "is", "what", "when", "which"}
    terms = set(re.findall(r"\w+", question.lower())) - stopwords
    matches = []
    for path in paths:
        content = read_local_text(path)
        score = sum(len(re.findall(r"\b" + re.escape(term) + r"\b", content.lower())) for term in terms)
        if score:
            matches.append((score, str(path.relative_to(DATA_ROOT)), content[:250]))
    return sorted(matches, key=lambda x: (-x[0], x[1]))[:top_k]

print(keyword_search("well drilling report", files))

## Validate your own answers CSV

The example search results are **not answers** and must not be treated as verified evidence. Use the 32 questions in `DATA_ROOT / "open_test/open_test_questions.csv"` and your own QA workflow to generate predictions. Copy the supplied `track2_predictions_template.csv`, keep all question IDs, and save it as your team's `submission/predictions.csv`. Judges accept equivalent correct wording and review source-file/page evidence under the Submission Guide rubric.

In [ ]:
PREDICTIONS = Path("submission/predictions.csv")  # set after implementing your solution
if PREDICTIONS.is_file():
    with PREDICTIONS.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
        print("Exact output header:", reader.fieldnames == EXPECTED_COLUMNS)
        print("Rows:", len(rows))
        print("Duplicate question IDs:", len(rows) - len(set(row.get("question_id") for row in rows)))
        print("Answers without citations (review abstentions separately):", sum(bool(row.get("answer", "").strip()) and not row.get("evidence", "").strip() for row in rows))
else:
    print("Validator skipped: generate your own answers CSV first.")

## Your implementation

Add approved PDF parsing, retrieval or extraction, grounded answering, evidence paths, abstention and confidence. Save any fitted indexes/models before submission. Ensure your submitted notebook can reproduce its outputs using the approved archive and declared dependencies.

## Validate Your Submission (optional)

Set `SUBMISSION_DIR` to your private team repository's `submission/` folder, then run both cells. Checks formatting and required files, not answer quality or solution execution.

In [ ]:
import csv
import json
import math
from pathlib import Path

HEADERS = {
    "1": ("page_id", "well", "depth_m", "curve", "value", "confidence"),
    "2": ("question_id", "answer", "evidence", "confidence"),
}

def validate_structure(folder):
    folder = Path(folder)
    errors = []
    for name in ("README.md", "requirements.txt", "predictions.csv"):
        path = folder / name
        if not path.is_file() or not path.read_text(encoding="utf-8").strip():
            errors.append("Missing or empty required file: " + name)
    notebooks = sorted(folder.glob("*.ipynb"))
    if not notebooks:
        errors.append("At least one Jupyter notebook (.ipynb) is required")
    for notebook in notebooks:
        try:
            data = json.loads(notebook.read_text(encoding="utf-8"))
            if not isinstance(data.get("nbformat"), int) or not isinstance(data.get("cells"), list):
                errors.append("Invalid notebook structure: " + notebook.name)
        except (json.JSONDecodeError, UnicodeDecodeError):
            errors.append("Unreadable notebook: " + notebook.name)
    return {"valid": not errors, "errors": errors, "notebooks": len(notebooks)}

def validate_csv(path, track):
    path = Path(path)
    errors = []
    if not path.is_file():
        return {"valid": False, "errors": ["Predictions CSV not found"], "rows": 0}
    try:
        with path.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            columns = reader.fieldnames or []
            missing = [name for name in HEADERS[track] if name not in columns]
            if missing:
                errors.append("Missing columns: " + ", ".join(missing))
            if len(columns) != len(set(columns)):
                errors.append("Duplicate CSV column headers")
            rows = list(reader)
    except (UnicodeDecodeError, csv.Error) as exc:
        return {"valid": False, "errors": [f"Unreadable CSV: {exc}"], "rows": 0}
    keys = ("page_id", "depth_m", "curve") if track == "1" else ("question_id",)
    seen = set()
    for number, row in enumerate(rows, 2):
        if None in row or any(v is None for v in row.values()):
            errors.append(f"Line {number}: incorrect number of columns")
            continue
        key = tuple((row.get(k) or "").strip() for k in keys)
        if not all(key):
            errors.append(f"Line {number}: missing identifier")
        elif key in seen:
            errors.append(f"Line {number}: duplicate identifier")
        seen.add(key)
        if track == "1":
            for field in ("depth_m", "value"):
                try:
                    value = float(row.get(field, ""))
                    if not math.isfinite(value):
                        errors.append(f"Line {number}: non-finite {field}")
                except (ValueError, TypeError):
                    errors.append(f"Line {number}: invalid {field}")
        if (row.get("confidence") or "").strip():
            try:
                confidence = float(row["confidence"])
                if not math.isfinite(confidence) or not 0 <= confidence <= 1:
                    errors.append(f"Line {number}: confidence must be between 0 and 1")
            except ValueError:
                errors.append(f"Line {number}: invalid confidence")
    if not rows:
        errors.append("CSV has no data rows")
    return {"valid": not errors, "errors": errors, "rows": len(rows)}




In [ ]:
# Example only: replace pipeline-x and adjust for your current working directory.
SUBMISSION_DIR = Path("../nape-aice-2026-pipeline-x/submission")
report = {"structure": validate_structure(SUBMISSION_DIR), "csv": validate_csv(SUBMISSION_DIR / "predictions.csv", "2")}
print(json.dumps(report, indent=2))
print("PASS: format valid" if all(x["valid"] for x in report.values()) else "FAIL: fix errors above")
